# Build  and  AI computer science TUTOR

### In this project I will build an AI computer science tutor using and pretrained model

## Install the Libraries

In [ ]:
#!pip install transformers datasets accelerate

### Load the Pretrained Model
 we will work with DistilGPT2 because it is small and fast

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(model_name)

In [ ]:
print("Model loaded successfully")
print(type(model))
print(type(tokenizer))

### let's test the model before starting

In [ ]:
# from transformers import pipeline, set_seed

# tokenizer.pad_token = tokenizer.eos_token
# model.config.pad_token_id = tokenizer.eos_token_id

# generator = pipeline(
#     "text-generation",
#     model=model,
#     tokenizer=tokenizer
# )

# set_seed(42)

# prompt = "Explain what a stack data structure is in computer science."

# result = generator(
#     prompt,
#     max_new_tokens=60,
#     do_sample=True,
#     temperature=0.7,
#     pad_token_id=tokenizer.eos_token_id
# )

# print(result[0]["generated_text"])

In [ ]:


# import os
# os.makedirs("/content/drive/MyDrive/AI_Projects/CS_Tutor_AI/model", exist_ok=True)

# save_path = "/content/drive/MyDrive/AI_Projects/CS_Tutor_AI/model/base_model"

# model.save_pretrained(save_path)
# tokenizer.save_pretrained(save_path)

# print("Base model saved successfully at:", save_path)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

path = "/content/drive/MyDrive/AI_Projects/CS_Tutor_AI/model/base_model"

tokenizer = AutoTokenizer.from_pretrained(path)
model = AutoModelForCausalLM.from_pretrained(path)

In [ ]:
import os

base_path = "/content/drive/MyDrive/AI_Projects/CS_Tutor_AI/datasets"

folders = [
    "raw/eli5",
    "raw/stackoverflow",
    "custom",
    "merged"
]

for folder in folders:
    os.makedirs(os.path.join(base_path, folder), exist_ok=True)

print("Dataset folders created successfully.")

In [ ]:
from datasets import load_dataset

eli5 = load_dataset("dany0407/eli5_category")

print(eli5)
print(eli5["train"][0])

In [ ]:
print(eli5)
print(eli5["train"][0])

In [ ]:
eli5.save_to_disk(
    "/content/drive/MyDrive/AI_Projects/CS_Tutor_AI/datasets/raw/eli5"
)

print("ELI5 dataset saved successfully.")

## Load stack Overflow data

In [ ]:
from datasets import load_dataset

stack_data = load_dataset("PrimeIntellect/stackexchange-question-answering")

print(stack_data)
print(stack_data["train"][0])

In [ ]:
stack_data.save_to_disk(
    "/content/drive/MyDrive/AI_Projects/CS_Tutor_AI/datasets/raw/stackexchange_primeintellect"
)

print("PrimeIntellect Stack Exchange dataset saved successfully.")

## Reload both datasets from Drive

In [ ]:
from datasets import load_from_disk

eli5_path = "/content/drive/MyDrive/AI_Projects/CS_Tutor_AI/datasets/raw/eli5"
stack_path = "/content/drive/MyDrive/AI_Projects/CS_Tutor_AI/datasets/raw/stackexchange_primeintellect"

eli5 = load_from_disk(eli5_path)
stack_data = load_from_disk(stack_path)

print("ELI5 loaded:", eli5)
print("StackExchange loaded:", stack_data)

## Inspect one example from each dataset

In [ ]:
print("\n===== ELI5 SAMPLE =====")
print(eli5["train"][0])

print("\n===== STACKEXCHANGE SAMPLE =====")
print(stack_data["train"][0])

## Create folders for cleaned datasets

In [ ]:
import os

cleaned_base = "/content/drive/MyDrive/AI_Projects/CS_Tutor_AI/datasets/cleaned"
os.makedirs(cleaned_base, exist_ok=True)
os.makedirs(f"{cleaned_base}/eli5_cs", exist_ok=True)
os.makedirs(f"{cleaned_base}/stackexchange_cs", exist_ok=True)

print("Cleaned dataset folders created.")

## Prepare CS keywords

In [ ]:
cs_keywords = [
    "computer science", "programming", "coding", "algorithm", "algorithms",
    "data structure", "data structures", "stack", "queue", "linked list",
    "array", "arraylist", "hash table", "hashmap", "tree", "binary tree",
    "graph", "dfs", "bfs", "recursion", "dynamic programming", "sorting",
    "searching", "binary search", "linear search", "big o", "time complexity",
    "space complexity", "object oriented", "oop", "class", "object",
    "inheritance", "polymorphism", "encapsulation", "interface",
    "database", "sql", "primary key", "foreign key", "normalization",
    "operating system", "os", "compiler", "interpreter", "cpu", "ram",
    "thread", "process", "network", "http", "https", "api", "git", "github",
    "python", "java", "javascript", "c++", "c#", "html", "css", "react",
    "fastapi", "machine learning", "artificial intelligence", "ai"
]

## Clean and filter ELI5

In [ ]:
def eli5_to_qa(example):
    question = example["title"].strip() if example["title"] else ""

    answer_list = example["answers"]["text"] if "text" in example["answers"] else []
    answer = answer_list[0].strip() if len(answer_list) > 0 and answer_list[0] else ""

    full_question = question.lower()
    is_cs = any(keyword in full_question for keyword in cs_keywords)

    return {
        "question": question,
        "answer": answer,
        "is_cs": is_cs,
        "source": "eli5"
    }

eli5_train = eli5["train"].map(eli5_to_qa)

eli5_cs = eli5_train.filter(
    lambda x: x["is_cs"] and len(x["question"]) > 15 and len(x["answer"]) > 50
)

print("Original ELI5 train size:", len(eli5["train"]))
print("Filtered ELI5 CS size:", len(eli5_cs))
print(eli5_cs[0])

## Clean and filter StackExchange

In [ ]:
def stack_to_qa(example):
    question = example["prompt"].strip() if example["prompt"] else ""
    answer = example["gold_standard_solution"].strip() if example["gold_standard_solution"] else ""

    full_question = question.lower()
    is_cs = any(keyword in full_question for keyword in cs_keywords)

    return {
        "question": question,
        "answer": answer,
        "is_cs": is_cs,
        "source": "stackexchange"
    }

stack_train = stack_data["train"].map(stack_to_qa)

stack_cs = stack_train.filter(
    lambda x: x["is_cs"] and len(x["question"]) > 15 and len(x["answer"]) > 50
)

print("Original StackExchange train size:", len(stack_data["train"]))
print("Filtered StackExchange CS size:", len(stack_cs))
print(stack_cs[0])

## Save the cleaned datasets

In [ ]:
eli5_cs.save_to_disk(f"{cleaned_base}/eli5_cs")
stack_cs.save_to_disk(f"{cleaned_base}/stackexchange_cs")

print("Cleaned CS datasets saved successfully.")

## Turn them into training text format

In [ ]:
def format_training_text(example):
    text = f"Question: {example['question']}\nAnswer: {example['answer']}"
    return {"text": text}

eli5_cs_formatted = eli5_cs.map(format_training_text)
stack_cs_formatted = stack_cs.map(format_training_text)

print(eli5_cs_formatted[0]["text"])
print()
print(stack_cs_formatted[0]["text"])

## Save formatted versions too

In [ ]:
os.makedirs(f"{cleaned_base}/eli5_cs_formatted", exist_ok=True)
os.makedirs(f"{cleaned_base}/stackexchange_cs_formatted", exist_ok=True)

eli5_cs_formatted.save_to_disk(f"{cleaned_base}/eli5_cs_formatted")
stack_cs_formatted.save_to_disk(f"{cleaned_base}/stackexchange_cs_formatted")

print("Formatted CS datasets saved successfully.")

## Load the formatted datasets

In [ ]:
from datasets import load_from_disk, concatenate_datasets
import os

cleaned_base = "/content/drive/MyDrive/AI_Projects/CS_Tutor_AI/datasets/cleaned"
merged_base = "/content/drive/MyDrive/AI_Projects/CS_Tutor_AI/datasets/merged"

os.makedirs(merged_base, exist_ok=True)

eli5_cs_formatted = load_from_disk(f"{cleaned_base}/eli5_cs_formatted")
stack_cs_formatted = load_from_disk(f"{cleaned_base}/stackexchange_cs_formatted")

print("ELI5 formatted size:", len(eli5_cs_formatted))
print("StackExchange formatted size:", len(stack_cs_formatted))

## Sample StackExchange down to a balanced size

In [ ]:
stack_sample_size = 50000

if len(stack_cs_formatted) > stack_sample_size:
    stack_cs_sampled = stack_cs_formatted.shuffle(seed=42).select(range(stack_sample_size))
else:
    stack_cs_sampled = stack_cs_formatted

print("Sampled StackExchange size:", len(stack_cs_sampled))

## Merge ELI5 + sampled StackExchange

In [ ]:
merged_dataset = concatenate_datasets([eli5_cs_formatted, stack_cs_sampled])
merged_dataset = merged_dataset.shuffle(seed=42)

print("Merged dataset size before dedup:", len(merged_dataset))
print(merged_dataset[0])

## Remove duplicate text rows

In [ ]:
seen = set()
unique_indices = []

for i, item in enumerate(merged_dataset):
    text = item["text"].strip()
    if text not in seen:
        seen.add(text)
        unique_indices.append(i)

merged_dataset_dedup = merged_dataset.select(unique_indices)

print("Merged dataset size after dedup:", len(merged_dataset_dedup))

In [ ]:
columns_to_keep = ["question", "answer", "source", "text"]

merged_dataset_dedup = merged_dataset_dedup.remove_columns(
    [col for col in merged_dataset_dedup.column_names if col not in columns_to_keep]
)

print("Remaining columns:", merged_dataset_dedup.column_names)
print(merged_dataset_dedup[0])

### Save the merged dataset

In [ ]:
merged_path = f"{merged_base}/merged_v1_no_custom"
merged_dataset_dedup.save_to_disk(merged_path)

print("Merged dataset saved successfully at:", merged_path)

In [ ]:
# let's inspect these datasets
for i in range(3):
    print(f"\n===== SAMPLE {i+1} =====")
    print(merged_dataset_dedup[i]["text"][:1000])

## Create our custom datasets

In [ ]:
import os

custom_base = "/content/drive/MyDrive/AI_Projects/CS_Tutor_AI/datasets/custom"

os.makedirs(custom_base, exist_ok=True)

print("Custom dataset folder ready:", custom_base)

### Create the topic plan

In [ ]:
topics = {
    "programming_basics": [
        "variables", "data types", "operators", "if statements", "loops",
        "functions", "parameters", "return values", "scope", "debugging"
    ],
    "oop": [
        "class", "object", "constructor", "inheritance", "polymorphism",
        "encapsulation", "abstraction", "interface", "method overloading", "method overriding"
    ],
    "data_structures": [
        "array", "arraylist", "linked list", "stack", "queue",
        "hash table", "binary tree", "heap", "graph", "set"
    ],
    "algorithms": [
        "linear search", "binary search", "bubble sort", "merge sort", "quick sort",
        "recursion", "dynamic programming", "greedy algorithm", "dfs", "bfs"
    ],
    "complexity": [
        "big o notation", "time complexity", "space complexity", "o(1)", "o(log n)",
        "o(n)", "o(n log n)", "o(n^2)", "best case", "worst case"
    ],
    "databases": [
        "database", "table", "row", "column", "primary key",
        "foreign key", "sql join", "index", "normalization", "transaction"
    ],
    "systems": [
        "operating system", "process", "thread", "cpu", "ram",
        "cache", "compiler", "interpreter", "file system", "virtual memory"
    ],
    "networking": [
        "ip address", "dns", "http", "https", "tcp", "udp",
        "client server", "api", "latency", "packet"
    ],
    "software_engineering": [
        "git", "github", "version control", "testing", "unit testing",
        "integration testing", "debugging tools", "frontend", "backend", "deployment"
    ],
    "ai_ml_basics": [
        "machine learning", "supervised learning", "unsupervised learning", "neural network",
        "training data", "validation data", "overfitting", "underfitting", "gradient descent", "classification"
    ]
}

total_topics = sum(len(v) for v in topics.values())
print("Number of categories:", len(topics))
print("Total base topics:", total_topics)

## Create question templates

In [ ]:
question_templates = [
    "What is {topic} in computer science?",
    "Explain {topic} in simple words.",
    "Explain {topic} like I am a beginner.",
    "Why is {topic} important in computer science?",
    "What is the difference between {topic} and a related concept?",
    "When should you use {topic}?",
    "What problem does {topic} solve?",
    "Give a simple example of {topic}.",
    "How does {topic} work?",
    "What are the advantages and disadvantages of {topic}?"
]

print("Number of question templates:", len(question_templates))

## Build a starter custom dataset skeleton

In [ ]:
custom_examples = []

for category, topic_list in topics.items():
    for topic in topic_list:
        for template in question_templates:
            custom_examples.append({
                "category": category,
                "topic": topic,
                "question": template.format(topic=topic),
                "answer": "",
                "source": "custom"
            })

print("Total custom skeleton examples:", len(custom_examples))
print(custom_examples[:5])

## Save the skeleton as JSON

In [ ]:
import json
import os

skeleton_path = os.path.join(custom_base, "custom_cs_skeleton_1000.json")

with open(skeleton_path, "w", encoding="utf-8") as f:
    json.dump(custom_examples, f, indent=2, ensure_ascii=False)

print("Custom skeleton saved to:", skeleton_path)

## Create a small high-quality answer bank first

In [ ]:
answer_bank = {
    "array": "An array is a data structure that stores multiple values in a fixed-size sequence. Each item is stored next to each other in memory, and each position has an index. Arrays are fast when you want to access an element by its index, but adding or removing items in the middle can be slow.",

    "linked list": "A linked list is a data structure made of nodes. Each node stores data and a reference to the next node. Unlike an array, linked list elements are not stored in one continuous block of memory. Linked lists make insertion and deletion easier in some cases, but accessing an item by position is slower.",

    "stack": "A stack is a linear data structure that follows the Last In, First Out rule, also called LIFO. This means the last item added is the first one removed. Common operations are push, pop, and peek. A stack is useful for function calls, undo operations, and expression evaluation.",

    "queue": "A queue is a linear data structure that follows the First In, First Out rule, also called FIFO. The first item added is the first one removed. Common operations are enqueue, dequeue, and front. Queues are useful for scheduling, buffering, and task processing.",

    "hash table": "A hash table is a data structure that stores key-value pairs. It uses a hash function to convert a key into an index where the value is stored. Hash tables are very fast for searching, inserting, and deleting in average cases, but collisions can happen when two keys map to the same index.",

    "binary search": "Binary search is an algorithm used to find a value in a sorted list. It works by checking the middle element and cutting the search space in half each time. This makes it much faster than checking one element at a time. Its time complexity is O(log n).",

    "recursion": "Recursion is a technique where a function calls itself to solve a smaller version of the same problem. A recursive solution usually has a base case, which stops the recursion, and a recursive case, which keeps the process going. It is useful for trees, divide-and-conquer algorithms, and mathematical problems.",

    "big o notation": "Big O notation describes how the running time or memory usage of an algorithm grows as the input size increases. It helps compare algorithms by focusing on growth rather than exact time. For example, O(1) means constant time, O(n) means linear growth, and O(log n) means logarithmic growth.",

    "primary key": "A primary key is a column, or group of columns, that uniquely identifies each row in a database table. It cannot contain duplicate values, and it usually cannot be null. Primary keys help keep data organized and make it easier to connect tables.",

    "foreign key": "A foreign key is a column in one table that refers to the primary key of another table. It is used to create relationships between tables. Foreign keys help maintain data consistency and connect related information across the database.",

    "process": "A process is a program that is currently running. It has its own memory space and system resources. Processes are isolated from each other, which improves safety, but communication between them can be slower than communication between threads.",

    "thread": "A thread is a smaller unit of execution inside a process. Multiple threads in the same process share memory and resources. Threads can make programs faster and more responsive, but they can also create problems like race conditions if not managed carefully.",

    "compiler": "A compiler translates source code from a programming language into machine code before the program runs. This allows the computer to execute the program directly. Compiled programs are often faster, but the compilation step happens before execution.",

    "interpreter": "An interpreter reads and executes code line by line at runtime instead of translating the whole program first. This makes testing and debugging easier, but interpreted programs can be slower than compiled ones.",

    "http": "HTTP stands for HyperText Transfer Protocol. It is the set of rules used for communication between a web browser and a web server. It defines how requests and responses are sent over the web. HTTP itself is not encrypted.",

    "https": "HTTPS is the secure version of HTTP. It uses encryption, usually through TLS or SSL, to protect data while it is being sent between the browser and the server. HTTPS helps keep passwords, payment details, and private data safe.",

    "git": "Git is a version control system used to track changes in code over time. It allows developers to save versions of a project, create branches, merge work, and collaborate with others. Git helps teams manage code safely and efficiently.",

    "machine learning": "Machine learning is a field of artificial intelligence where computers learn patterns from data instead of being programmed with every rule directly. A model is trained using examples, then used to make predictions or decisions on new data.",

    "supervised learning": "Supervised learning is a type of machine learning where the model learns from labeled data. This means each training example includes both the input and the correct output. It is commonly used for classification and regression tasks.",

    "neural network": "A neural network is a machine learning model inspired by the way neurons connect in the brain. It is made of layers of nodes that process data and learn patterns through training. Neural networks are powerful for tasks like image recognition, language processing, and prediction."
}

print("Answer bank topics:", len(answer_bank))
print(answer_bank.keys())

## Auto-fill the examples

In [ ]:
for ex in custom_examples:
    topic = ex["topic"]
    if topic in answer_bank:
        ex["answer"] = answer_bank[topic]

filled_count = sum(1 for ex in custom_examples if ex["answer"].strip())
empty_count = len(custom_examples) - filled_count

print("Filled examples:", filled_count)
print("Still empty:", empty_count)

## Save this first custom version

In [ ]:
custom_v1_path = os.path.join(custom_base, "custom_cs_v1_partial.json")

with open(custom_v1_path, "w", encoding="utf-8") as f:
    json.dump(custom_examples, f, indent=2, ensure_ascii=False)

print("Partial custom dataset saved to:", custom_v1_path)

## illed examples into training format

In [ ]:
custom_text_dataset = []

for ex in custom_examples:

    if ex["answer"].strip() == "":
        continue

    text = f"""Question: {ex['question']}

Answer: {ex['answer']}"""

    custom_text_dataset.append({
        "question": ex["question"],
        "answer": ex["answer"],
        "source": "custom",
        "text": text
    })

print("Custom examples ready for training:", len(custom_text_dataset))
print(custom_text_dataset[0])

## Save

In [ ]:
from datasets import Dataset

custom_dataset = Dataset.from_list(custom_text_dataset)

custom_dataset_path = custom_base + "/custom_cs_v1_dataset"

custom_dataset.save_to_disk(custom_dataset_path)

print("Custom dataset saved at:", custom_dataset_path)

# Load both datasets

In [ ]:
from datasets import load_from_disk, concatenate_datasets

main_merged_path = "/content/drive/MyDrive/AI_Projects/CS_Tutor_AI/datasets/merged/merged_v1_no_custom"
custom_dataset_path = "/content/drive/MyDrive/AI_Projects/CS_Tutor_AI/datasets/custom/custom_cs_v1_dataset"

main_dataset = load_from_disk(main_merged_path)
custom_dataset = load_from_disk(custom_dataset_path)

print("Main dataset size:", len(main_dataset))
print("Custom dataset size:", len(custom_dataset))
print("Main columns:", main_dataset.column_names)
print("Custom columns:", custom_dataset.column_names)

# Merge them

In [ ]:
final_dataset = concatenate_datasets([main_dataset, custom_dataset])
final_dataset = final_dataset.shuffle(seed=42)

print("Final dataset size:", len(final_dataset))
print(final_dataset[0])

# Save the final dataset

In [ ]:

final_dataset_path = "/content/drive/MyDrive/AI_Projects/CS_Tutor_AI/datasets/merged/final_dataset_v1"

final_dataset.save_to_disk(final_dataset_path)

print("Final merged dataset saved at:", final_dataset_path)

# Create train/validation split

In [ ]:
split_dataset = final_dataset.train_test_split(test_size=0.1, seed=42)

train_dataset = split_dataset["train"]
val_dataset = split_dataset["test"]

print("Train size:", len(train_dataset))
print("Validation size:", len(val_dataset))

# Save the split datasets

In [ ]:
train_path = "/content/drive/MyDrive/AI_Projects/CS_Tutor_AI/datasets/merged/train_dataset_v1"
val_path = "/content/drive/MyDrive/AI_Projects/CS_Tutor_AI/datasets/merged/val_dataset_v1"

train_dataset.save_to_disk(train_path)
val_dataset.save_to_disk(val_path)

print("Train dataset saved at:", train_path)
print("Validation dataset saved at:", val_path)

# Load model and tokenizer

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_from_disk

model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

train_dataset = load_from_disk("/content/drive/MyDrive/AI_Projects/CS_Tutor_AI/datasets/merged/train_dataset_v1")
val_dataset = load_from_disk("/content/drive/MyDrive/AI_Projects/CS_Tutor_AI/datasets/merged/val_dataset_v1")

print("Model and datasets loaded successfully.")
print("Train size:", len(train_dataset))
print("Validation size:", len(val_dataset))

# Tokenize the text

In [ ]:
max_length = 256

def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=max_length
    )

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)

print("Tokenization complete.")
print(tokenized_train[0].keys())

# Create labels for causal language modeling

In [ ]:
def add_labels(example):
    example["labels"] = example["input_ids"].copy()
    return example

tokenized_train = tokenized_train.map(add_labels)
tokenized_val = tokenized_val.map(add_labels)

columns_to_keep = ["input_ids", "attention_mask", "labels"]

tokenized_train = tokenized_train.remove_columns(
    [col for col in tokenized_train.column_names if col not in columns_to_keep]
)

tokenized_val = tokenized_val.remove_columns(
    [col for col in tokenized_val.column_names if col not in columns_to_keep]
)

print("Labels added and columns cleaned.")
print(tokenized_train[0])

# Set training arguments

In [ ]:
from transformers import TrainingArguments

output_dir = "/content/drive/MyDrive/AI_Projects/CS_Tutor_AI/model/cs_tutor_model_v1"

training_args = TrainingArguments(
    output_dir=output_dir,
    do_train=True,
    do_eval=True,
    logging_steps=100,
    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    save_total_limit=2,
    fp16=True,
    report_to=[]
)

print("Training arguments ready.")

# Create the trainer

In [ ]:
from transformers import Trainer, DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator
)

print("Trainer created successfully.")

# Start training

In [ ]:
trainer.train()

# Save the final fine-tuned model

In [ ]:
final_model_path = "/content/drive/MyDrive/AI_Projects/CS_Tutor_AI/model/cs_tutor_model_v1_final"

trainer.save_model(final_model_path)
tokenizer.save_pretrained(final_model_path)

print("Fine-tuned model saved at:", final_model_path)

## Load the fine-tuned model

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_path = "/content/drive/MyDrive/AI_Projects/CS_Tutor_AI/model/cs_tutor_model_v1_final"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

print("CS Tutor model loaded successfully.")

## Create a generation function

In [ ]:
def ask_cs_tutor(question, max_tokens=120):

    prompt = f"Question: {question}\n\nAnswer:"

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(device)

    output = model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    answer = tokenizer.decode(output[0], skip_special_tokens=True)

    print("\n-----------------------------------")
    print(answer)
    print("-----------------------------------\n")

In [ ]:
ask_cs_tutor("What is a stack in computer science?")

In [ ]:
ask_cs_tutor("Explain recursion like I am a beginner.")

In [ ]:
ask_cs_tutor("What is the difference between a process and a thread?")

In [ ]:
ask_cs_tutor("What is Big O notation?")

# Add a search tool to make the model strong

## Install a search library

In [ ]:
#!pip install sentence-transformers faiss-cpu

## Load your dataset text

In [ ]:
from datasets import load_from_disk

dataset_path = "/content/drive/MyDrive/AI_Projects/CS_Tutor_AI/datasets/merged/train_dataset_v1"

dataset = load_from_disk(dataset_path)

texts = dataset["text"]

print("Loaded texts:", len(texts))

## Create embeddings

In [ ]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = embed_model.encode(texts, show_progress_bar=True)

## Build the FAISS search index

In [ ]:
import faiss
import numpy as np

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

print("FAISS index ready.")

## Create a smarter tutor function

In [ ]:
def ask_cs_tutor(question, k=3):
    # Embed the question
    q_embedding = embed_model.encode([question])

    # Search similar examples
    distances, indices = index.search(np.array(q_embedding), k)

    retrieved_examples = []
    for idx in indices[0]:
        idx = int(idx)
        retrieved_examples.append(dataset[idx])

    # Build a cleaner context using only the retrieved answers
    context = ""
    for i, ex in enumerate(retrieved_examples, 1):
        context += f"Example {i}\n"
        context += f"Question: {ex['question']}\n"
        context += f"Answer: {ex['answer']}\n\n"

    prompt = f"""
You are a computer science tutor.

Below are explanations taken from trusted CS discussions.

Use them to answer the question clearly.

If the answer appears in the examples, explain it in simple words.
Do not invent unrelated information.

Examples:
{context}

Question: {question}

Answer in 3 short sentences:
"""

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(device)

    output = model.generate(
    **inputs,
    max_new_tokens=60,
    do_sample=False,
    repetition_penalty=1.4,
    num_beams=3,
    early_stopping=True,
    pad_token_id=tokenizer.eos_token_id
)

    # full_output = tokenizer.decode(output[0], skip_special_tokens=True)

    # # Try to show only the final tutor answer
    # if "Tutor answer:" in full_output:
    #     answer = full_output.split("Tutor answer:")[-1].strip()
    # else:
    #     answer = full_output.strip()
    full_output = tokenizer.decode(output[0], skip_special_tokens=True)

    if "Answer in 3 short sentences:" in full_output:
        answer = full_output.split("Answer in 3 short sentences:")[-1].strip()
    else:
        answer = full_output.strip()

    print("\n----------------------------")
    print("Question:", question)
    print("\nAnswer:", answer)
    print("----------------------------\n")

## Test

In [ ]:
ask_cs_tutor("What is a stack in computer science?")

In [ ]:
ask_cs_tutor("Explain recursion like I am a beginner.")

In [ ]:
ask_cs_tutor("What is the difference between a process and a thread?")

# Add a simple Gradio web app

In [ ]:
#!pip install gradio

In [ ]:
import gradio as gr

def cs_tutor_app(question):
    q_embedding = embed_model.encode([question])
    distances, indices = index.search(np.array(q_embedding), 3)

    context = ""
    for idx in indices[0]:
        idx = int(idx)
        context += texts[idx] + "\n\n"

    prompt = f"""
You are a helpful computer science tutor.

Use the examples below to answer the user's question.
Write a short, clear, beginner-friendly answer.
Do not repeat the examples word for word.
Do not invent unrelated information.
Keep the answer to 3 or 4 sentences maximum.

{context}

User question: {question}

Tutor answer:
"""

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(device)

    output = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=False,
        repetition_penalty=1.4,
        num_beams=3,
        early_stopping=True,
        pad_token_id=tokenizer.eos_token_id
    )

    full_output = tokenizer.decode(output[0], skip_special_tokens=True)

    if "Tutor answer:" in full_output:
        answer = full_output.split("Tutor answer:")[-1].strip()
    else:
        answer = full_output.strip()

    return answer

demo = gr.Interface(
    fn=cs_tutor_app,
    inputs=gr.Textbox(lines=2, placeholder="Ask a computer science question..."),
    outputs=gr.Textbox(label="Tutor Answer"),
    title="AI Computer Science Tutor",
    description="Ask beginner-friendly computer science questions and get short explanations."
)

demo.launch(share=True)